#### Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as mp
import seaborn as sea
import datetime as dt
dbd="database"
file="file"
path="path"
df=pd.read_csv(f"{path}\{file}.csv")
df=pd.DataFrame(df)
c1 = "date"
c2 = "time"

#### Clean Check

In [ ]:
# Null
df.isnull().sum()

# Describe
df.describe()

# Duplicate 
df.duplicated().sum()

# Null Check
df.isnull().mean() * 100

#### Column Date

In [ ]:
dates=[i for i in df.columns if any(j in i for j in ["date","time","dt","timestamp"])]
for i in dates:
    if pd.api.types.is_datetime64_any_dtype(df[i]):
        base=i.rstrip("_")
        df[f"{base}_day"]=df[i].dt.day
        df[f"{base}_month"]=df[i].dt.month
        df[f"{base}_year"]=df[i].dt.year

#### Column Date And Time

In [ ]:
df[c1] = df[c1].astype(str)
df[c2] = pd.to_datetime(df[c2], errors='coerce', utc=True).dt.time
df['Date_Time'] = pd.to_datetime(df[c1].astype(str) + ' ' + df[c2].astype(str), errors='coerce')

#### Column Date And Time Format

In [ ]:
for col in df.columns:
    if pd.api.types.is_datetime64_any_dtype(df[col]):
        df[col] = df[col].dt.strftime('%d-%m-%Y %I:%M:%S %p')

#### Column Date Format

In [ ]:
date_cols=[f'{c1}',f'{c2}']
for col in date_cols:
    temp = df[col].astype(str).str.split('-', expand=True)
    df[col] = pd.to_datetime(
        temp[2] + '-' + temp[1] + '-' + temp[0],  # year-month-day
        errors='coerce'
    )

#### Column Order

In [ ]:
first_cols = [f'{c1}',f'{c2}']
other_cols = [c for c in df.columns if c not in first_cols]
new_order = first_cols + other_cols
df = df[new_order]

#### Column Quantity

In [ ]:
for col in df.columns:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False).sort_index(ascending=True))
    print("\n")

#### Column Table

In [ ]:
# df = pd.read_sql("select * from payment_data", db)
# df.head()

#### Column Rename

In [ ]:
cr1="n1"
cr2="n2"
df.rename(columns={
    f'{c1}': f'{cr1}',
    f'{c2}': f'{cr2}'
},inplace=True)

#### Column Delete

In [ ]:
df = df.drop(columns=[f'{c1}', f'{c2}'])

#### Column Same Column

In [ ]:
df = df.loc[:,~df.T.duplicated()]

#### Column Display

In [ ]:
df_renamed = df.rename(columns={col: f"{col} (c{i+1})" for i, col in enumerate(df.columns)})
display(
    df.head()
    .style
    .hide(axis="index")
    .set_properties(**{'text-align': 'left'})
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]},
                       {'selector': 'th', 'props': [('border', '1.5px solid white')]},
                       {'selector': 'td', 'props': [('border', '1px solid white')]}             
    ])
)

#### Column Duplicate

In [ ]:
display(df[df.duplicated(subset=[f'{c1}',f'{c2}'], keep=False)])

#### Column Duplicate Unique 

In [ ]:
df.drop_duplicates(subset=[f'{c1}',f'{c2}'], inplace=True)

#### Column Int Datatype

In [ ]:
cols_to_convert = [f"{c1}",f"{c2}"]
df[cols_to_convert] = df[cols_to_convert].astype("int64")

#### Column Time

In [ ]:
time_columns = [f'{c2}']
for col in time_columns:
    df[col] = pd.to_datetime('1900-01-01 ' + df[col].astype(str), errors='coerce').astype('datetime64[us]')

#### Column Table Merge

In [ ]:
import os
folder = r"D:\ZZ\Clean"
df1 = pd.read_csv(os.path.join(folder, "titanic.csv"))
df2 = pd.read_csv(os.path.join(folder, "survive.csv"))
df = pd.merge(df1, df2, on='passengerid', how='inner')
df.head()

#### Column Table Concat

In [ ]:
import os
folder = r"D:\ZZ\Clean"
df1 = pd.read_csv(os.path.join(folder, "titanic.csv"))
df2 = pd.read_csv(os.path.join(folder, "survive.csv"))
df = pd.concat([df1, df2], join='inner',ignore_index=True)
df = df.drop_duplicates(subset='passengerid')
df.head()

#### Column Table Compare

In [ ]:
df1=df1.head()
df2=df2.head()
df = df1.compare(df2,align_axis=1,keep_equal=True,keep_shape=True)

#### Column Missing

In [ ]:
for col in df.columns: 
    if pd.api.types.is_datetime64_any_dtype(df[col]):
        df[col] = df[col].fillna(pd.Timestamp('1970-01-01'))
    elif pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(0)
    else:
            df[col] = df[col].fillna('missing')

# 30%

for col in df.columns:
    null_pct = df[col].isnull().mean()*100
    if null_pct < 30:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = df[col].ffill()
            df[col] = df[col].bfill()
        elif pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
    else:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = df[col].fillna(pd.Timestamp('1970-01-01'))
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(0)
        else:
            df[col] = df[col].fillna('missing')

#### Column Date Null

In [ ]:
df['new_column'] = '13:44:18'
df['new_column'] = pd.to_datetime(df['new_column'], errors='coerce', utc=True).dt.time

df[c1] = df[c1].astype(str)
df['_time1'] = pd.to_datetime(df[c1].astype(str) + ' ' + df['new_column'].astype(str), errors='coerce')

df = df.drop(columns=[f"{c1}_isnull","_time1"])
df.rename(columns={
    "_time1_isnull":f"{c1}_isnull"
},inplace=True)

#### Column Delete Null

In [ ]:
df = df.drop(columns=["date_isnull","time_isnull"])

#### Values Sorting

In [ ]:
df = df.sort_values(by=[f'{c1}',f'{c2}'], ascending=[True,False])
df = df.reset_index(drop=True)

#### Values Lower

In [ ]:
cols = [f'{c1}', f'{c2}']
df[cols] = df[cols].apply(lambda x: x.str.lower())

#### Values Title

In [ ]:
cols = [f'{c1}', f'{c2}']
df[cols] = df[cols].apply(lambda x: x.str.title())

#### Values Rename

In [ ]:
df[c1] = df[c1].str.lower()
df[c1] = df[c1].replace({
    'male': 'M',
    'female': 'F'
})

#### Values Negatives

In [ ]:
cols = [f'{c1}',f'{c2}']
for col in cols:
    df.loc[df[col] < 0, col] = None
    df[col] = df[col].fillna(df[col].mean())
    df[col] = df[col].astype(int)

#### Values Round

In [ ]:
df = df.round({
    f'{c1}': 1,
    f'{c2}':2
})

#### Values Floor

In [ ]:
cols = [f'{c1}',f'{c2}']
df[cols] = df[cols].apply(np.floor)

#### Values Ceil

In [ ]:
cols = [f'{c1}',f'{c2}'] 
df[cols] = df[cols].apply(np.ceil)

#### Values Limiting

In [ ]:
limits = {
    f'{c1}': (1, 100),
    f'{c2}':(1,500)
}
for col, (low, high) in limits.items():
    df.loc[~df[col].between(low, high), col] = None
    df[col] = df[col].fillna(df[col].median())

#### Values Limiting Equal

In [ ]:
cols_to_limit = [f'{c1}',f'{c2}']
for col in cols_to_limit:
    df[col] = df[col].str.extract(r'(\d+)')[0].astype('Int64')
    df = df[df[col].astype(str).str.len() == 7]

#### Values Split

In [ ]:
# Before
df[f'{c1}'] = df[f'{c1}'].str.split('[').str[0]

# After
df[f'{c1}'] = df[f'{c1}'].str.split('[').str[1]

# Inside
df[f'{c1}'] = df[f'{c1}'].str.extract(r'\[(.*?)\]')

# Outside
df[f'{c1}'] = df[f'{c1}'].str.extract(r'^(.*?)\[')

#### Values Merge

In [ ]:
df['Combined'] = df[c1].astype(str) + ' ' + df[c2].astype(str)

#### Values Merge Same

In [ ]:
df[c1] = df[c1].astype(str).apply(lambda x: ' '.join([part.strip() for part in x.split(',')[::-1]]))

# different
df['name'] = df[f'{c1}'].astype(str).apply(lambda x: ' '.join([part.strip() for part in x.split(',')[::-1]]))

#### Values Converting

In [ ]:
import re
def to_grams(x):
    x = str(x).lower()
    nums = re.findall(r'\d+\.?\d*', x)
    num = float(nums[0]) if nums else None
    
    if num is None:
        return None

    if 'kg' in x:
        value = num * 1000
    elif 'mg' in x:
        value = num / 1000
    else:
        value = num

    value = round(value, 1)
    return f"{value} g"
df[f'{c1}'] = df[f'{c1}'].apply(to_grams)

#### Values Unit Split

In [ ]:
df[['number', 'unit']] = df[f'{c1}'].str.extract(r'(\d+\.?\d*)\s*([a-zA-Z]*)')
df['number'] = df['number'].astype(float)

#### Values Remove

In [ ]:
df[c1] = df[c1].str.extract(r'(?:Mr\.|Mrs\.)\s*([^,()]+)')

#### Values Remove String In Number

In [ ]:
cols_to_clean = [f'{c1}',f'{c2}']
for col in cols_to_clean:
    df[col] = df[col].str.extract(r'(\d+)')[0].astype('Int64')
    df = df.dropna(subset=[col])


#### Values Remove Number In String

In [ ]:
cols_to_clean = [f'{c1}',f'{c2}']
for col in cols_to_clean:
    df[col] = df[col].astype(str).str.replace(r'\d+', '', regex=True)

#### Values Bool

In [ ]:
df[c1] = df[c1].notna().astype(int)

# Different
df['has cabin'] = df[f'{c1}'].notna().astype(int)

#### Values Null

In [ ]:
cols = [f'{c1}',f'{c2}']
df[cols] = df[cols].fillna('missing')

#### Values Correlation

In [ ]:
correlation = df['price'].corr(df['order_count'])
df = pd.DataFrame({
    'Metric': ['price vs order_count'],
    'Correlation': [correlation]
})

#### Values Whitespace

In [ ]:
df = df.replace('\n', ' ', regex=True)
df = df.replace(r'\s+', ' ', regex=True)

#### Values Special Character

In [ ]:
columns_to_clean = [f'{c1}',f'{c2}']
for col in columns_to_clean:
    clean = df[col].astype(str).str.replace(r"[^\d.-]+", "", regex=True)
    numeric_col = pd.to_numeric(clean, errors="coerce")
    if numeric_col.notna().sum() >= 0.6 * df[col].notna().sum():
        df[col] = numeric_col

#### Values Extra

In [ ]:
# Datetime
dates=[i for i in df.columns if any(j in i for j in ["date","dt","timestamp"])]
for col in dates:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Clean Folder
import os
import re
folder_name = "Clean"
clean_name = re.sub(r'[^A-Za-z0-9_-]', '_', folder_name)
os.makedirs(clean_name, exist_ok=True)

#### Values Duplicate

In [ ]:
df = df.drop_duplicates(subset="") 

#### Values Int

In [ ]:
df[df.select_dtypes(include='float64').columns] = \
    df.select_dtypes(include='float64').round().astype('Int64')